# 04 Train Context Models

This notebook trains the four context-aware NCF variants. The same split, negative samples, validation candidates, and hyperparameters are used for a fair ablation design.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.preprocessing import build_metadata_from_mappings
from src.train import train_model
from src.evaluate import evaluate_ranking
from src.analysis import append_or_replace_metrics
from src.utils import load_json, set_seed

set_seed()

In [ ]:
train_df = pd.read_csv(config.PROCESSED_DATA_DIR / 'train.csv')
val_df = pd.read_csv(config.PROCESSED_DATA_DIR / 'validation.csv')
val_candidates = pd.read_csv(config.PROCESSED_DATA_DIR / 'val_candidates.csv')
mappings = load_json(config.PROCESSED_DATA_DIR / 'mappings.json')
metadata = build_metadata_from_mappings(mappings)

## Model Variants

- `time`: adds hour, day of week, month, weekend flag, and time-of-day.
- `user`: adds gender, occupation, and age group.
- `movie`: adds genres and release year.
- `full`: combines all context groups.

In [ ]:
feature_names = {
    'time': 'user_id + movie_id + time context',
    'user': 'user_id + movie_id + user context',
    'movie': 'user_id + movie_id + movie context',
    'full': 'user_id + movie_id + time + user + movie context',
}

rows = []
histories = {}
for variant in ['time', 'user', 'movie', 'full']:
    print(f'\nTraining {variant}')
    model, history = train_model(variant, train_df, val_df, metadata, config.MODEL_SAVE_PATHS[variant])
    histories[variant] = history
    metrics, per_group, scored = evaluate_ranking(model, val_candidates, metadata, variant, k=config.TOP_K)
    row = {'split': 'validation', 'model': variant, 'used_features': feature_names[variant], **metrics}
    rows.append(row)
    display(pd.DataFrame([row]))

metrics_df = append_or_replace_metrics(config.RESULTS_DIR / 'metrics.csv', rows)
display(metrics_df.sort_values(['split', 'model']))

In [ ]:
for variant, history in histories.items():
    history.plot(x='epoch', y=['train_loss', 'val_loss'], marker='o', title=f'{variant} loss')
    plt.ylabel('BCE loss')
    plt.tight_layout()
    plt.show()